In [1]:
import os
import pandas as pd
import torch
from PIL import Image
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from tqdm import tqdm

/data/2_data_server/cv-07/anaconda3/envs/nlp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# -- 1. 설정 (Configuration) --
# 데이터 경로 설정
TEST_CSV_PATH = '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/dev_test.csv'
IMAGE_BASE_PATH = '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data'
SUBMISSION_CSV_PATH = './submission.csv' # 결과를 저장할 경로

# 모델 설정
MODEL_NAME = "Salesforce/blip2-opt-2.7b"

# 디바이스 설정 (GPU 우선 사용)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# -- 2. 모델 및 프로세서 로드 --
print(f"Loading model: {MODEL_NAME}...")
# Blip2Processor는 이미지 전처리, 텍스트 토큰화를 한번에 처리합니다.
processor = Blip2Processor.from_pretrained(MODEL_NAME)

# torch_dtype=torch.float16을 사용하여 모델을 반정밀도로 로드합니다.
# 이는 GPU 메모리 사용량을 줄이고 추론 속도를 향상시킵니다.
model = Blip2ForConditionalGeneration.from_pretrained(
    MODEL_NAME, 
    torch_dtype=torch.float16
)
model.to(DEVICE)
model.eval() # 모델을 평가 모드로 설정
print("Model loading complete.")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Using device: cuda
Loading model: Salesforce/blip2-opt-2.7b...


Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.63s/it]


Model loading complete.


In [3]:
# 전체 파라미터와 학습 가능한 파라미터 수를 계산
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"전체 파라미터: {total_params:,}개 ({total_params / 1_000_000:.2f}M)")
print(f"학습 가능한 파라미터: {trainable_params:,}개 ({trainable_params / 1_000_000:.2f}M)")

전체 파라미터: 3,744,761,856개 (3744.76M)
학습 가능한 파라미터: 3,744,761,856개 (3744.76M)


In [ ]:


# -- 3. 데이터 로드 --
df = pd.read_csv(TEST_CSV_PATH)
print(f"Test data loaded. Total questions: {len(df)}")

# -- 4. 추론 실행 --
results = []
# tqdm을 사용하여 진행 상황을 시각적으로 표시합니다.
for idx, row in tqdm(df.iterrows(), total=len(df)):
    
    # 이미지 경로 조합 및 이미지 로드
    # row['img_path']는 './input_images/TEST_XXX.jpg'와 같은 상대 경로이므로 os.path.join으로 절대 경로 생성
    img_path = os.path.join(IMAGE_BASE_PATH, row['img_path'].lstrip('./'))
    try:
        image = Image.open(img_path).convert("RGB")
    except FileNotFoundError:
        print(f"Error: Image not found at {img_path}")
        # 이미지를 찾을 수 없는 경우 -1을 답으로 기록하고 계속 진행
        results.append({'ID': row['ID'], 'answer': -1})
        continue

    # 질문과 선택지
    question = row['Question']
    choices = [row['A'], row['B'], row['C'], row['D']]
    
    choice_losses = []
    
    # 4개의 선택지에 대해 각각 loss 계산
    with torch.no_grad(): # 그래디언트 계산을 비활성화하여 메모리 사용량 줄이고 속도 향상
        for choice in choices:
            # VQA를 위한 프롬프트 구성
            prompt = f"Question: {question} Answer: {choice}"
            
            # 모델 입력 준비
            inputs = processor(
                images=image, 
                text=prompt, 
                return_tensors="pt"
            ).to(DEVICE, torch.float16)
            
            # 모델을 통해 loss 계산
            # labels 인자에 input_ids를 그대로 넣어주면, 모델이 내부적으로 정답 시퀀스에 대한 loss를 계산합니다.
            outputs = model(**inputs, labels=inputs.input_ids)
            loss = outputs.loss
            choice_losses.append(loss.item())

    # loss가 가장 낮은 선택지를 정답으로 선택 (1-based index)
    # choice_losses.index(min(choice_losses))는 0, 1, 2, 3 중 하나를 반환
    predicted_answer = choice_losses.index(min(choice_losses)) + 1
    results.append({'ID': row['ID'], 'answer': predicted_answer})

# -- 5. 제출 파일 생성 --
submission_df = pd.DataFrame(results)
submission_df.to_csv(SUBMISSION_CSV_PATH, index=False)

print(f"Inference complete. Submission file saved to {SUBMISSION_CSV_PATH}")
print("\n--- Submission Preview ---")
print(submission_df.head())
print("------------------------")